# 🏠 BDS Crawler — batdongsan.com.vn

**Purpose:** Collect property listing URLs then scrape detailed attributes  
  (price, area, address, coordinates, legal status, description …)  
  from [batdongsan.com.vn](https://batdongsan.com.vn).

**Pipeline**
```
Phase 1 — URL collection   → hrefs.txt
Phase 2 — Detail scraping  → data_bds.csv
Phase 3 — Description NLP  → data_bds_expanded.csv  (optional)
```

**Requirements:** `pip install -r requirements.txt`  
**Config:** copy `.env.example` → `.env` and adjust values before running.

## 1 · Install dependencies *(run once)*

In [8]:
# Run this cell only once per environment.
# After installation, restart the kernel before proceeding.
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "selenium>=4.18.0",
    "undetected-chromedriver>=3.5.5",
    "numpy>=1.26.0",
    "pandas>=2.2.0",
    "python-dotenv>=1.0.0",
    "setuptools<70.0.0",
    "packaging>=24.0",
])
print("✅ Dependencies installed — please restart the kernel.")

✅ Dependencies installed — please restart the kernel.


## 2 · Compatibility shim

In [9]:
# distutils was removed in Python 3.12; undetected-chromedriver may still
# import it internally via LooseVersion.  This shim keeps things working.
import sys
import types


class _LooseVersion:
    """Minimal LooseVersion replacement for Python 3.12+ compatibility."""

    def __init__(self, vstring: str) -> None:
        self.vstring = str(vstring)
        self.version = [
            int(x) if x.isdigit() else x
            for x in self.vstring.split(".")
        ]

    def __lt__(self, other: "_LooseVersion") -> bool:
        return self.version < other.version

    def __gt__(self, other: "_LooseVersion") -> bool:
        return self.version > other.version

    def __eq__(self, other: object) -> bool:
        return self.version == getattr(other, "version", other)

    def __ge__(self, other: "_LooseVersion") -> bool:
        return self.version >= other.version

    def __le__(self, other: "_LooseVersion") -> bool:
        return self.version <= other.version


_distutils = types.ModuleType("distutils")
_distutils.version = types.ModuleType("distutils.version")          # type: ignore[attr-defined]
_distutils.version.LooseVersion = _LooseVersion                     # type: ignore[attr-defined]
sys.modules.setdefault("distutils", _distutils)
sys.modules.setdefault("distutils.version", _distutils.version)     # type: ignore[arg-type]

<module 'distutils.version'>

## 3 · Imports & configuration

In [10]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import undetected_chromedriver as uc
from dotenv import load_dotenv
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# Load .env (silently skipped if the file does not exist)
load_dotenv(override=False)

# ── Configuration (override via .env) ────────────────────────────────────────
BASE_URL           = os.getenv("BASE_URL",           "https://batdongsan.com.vn/ban-nha-dat-tp-hcm")
QUERY_STRING       = os.getenv("QUERY_STRING",       "?vrs=1")
TARGET_SAMPLES     = int(os.getenv("TARGET_SAMPLES", 1))
PAGE_DELAY         = float(os.getenv("PAGE_DELAY",   2))
DETAIL_DELAY       = float(os.getenv("DETAIL_DELAY", 0.1))
CLOUDFLARE_TIMEOUT = int(os.getenv("CLOUDFLARE_TIMEOUT", 30))
ELEMENT_TIMEOUT    = int(os.getenv("ELEMENT_TIMEOUT",    15))
HREFS_FILE         = Path(os.getenv("HREFS_FILE",    "hrefs.txt"))
CSV_FILE           = Path(os.getenv("CSV_FILE",      "data_bds.csv"))
WINDOW_SIZE        = os.getenv("WINDOW_SIZE",        "1920,1080")
USER_AGENT         = os.getenv(
    "USER_AGENT",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36",
)

print("✅ Configuration loaded")
print(f"   BASE_URL       = {BASE_URL}")
print(f"   TARGET_SAMPLES = {TARGET_SAMPLES}")
print(f"   HREFS_FILE     = {HREFS_FILE}")
print(f"   CSV_FILE       = {CSV_FILE}")

✅ Configuration loaded
   BASE_URL       = https://batdongsan.com.vn/ban-nha-dat-tp-hcm
   TARGET_SAMPLES = 1
   HREFS_FILE     = hrefs.txt
   CSV_FILE       = data_bds.csv


## 4 · Pre-flight assertions

In [11]:
# Fail early if configuration values are obviously wrong.
assert BASE_URL.startswith("http"), f"BASE_URL must be a valid URL, got: {BASE_URL}"
assert TARGET_SAMPLES > 0,          f"TARGET_SAMPLES must be > 0, got: {TARGET_SAMPLES}"
assert PAGE_DELAY >= 0,             f"PAGE_DELAY must be >= 0, got: {PAGE_DELAY}"
assert DETAIL_DELAY >= 0,           f"DETAIL_DELAY must be >= 0, got: {DETAIL_DELAY}"
assert CLOUDFLARE_TIMEOUT > 0,      f"CLOUDFLARE_TIMEOUT must be > 0"
assert ELEMENT_TIMEOUT > 0,         f"ELEMENT_TIMEOUT must be > 0"

print("✅ All pre-flight assertions passed")

✅ All pre-flight assertions passed


## 5 · Chrome driver factory

In [12]:
def create_driver() -> uc.Chrome:
    """Return a new undetected-chromedriver instance.

    Configured for maximum compatibility with Cloudflare-protected sites.
    Does NOT use headless mode, which triggers bot-detection heuristics.
    """
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument(f"--window-size={WINDOW_SIZE}")
    options.add_argument(f"--user-agent={USER_AGENT}")
    return uc.Chrome(options=options)

## 6 · Scraping helpers

In [13]:
def wait_for_cloudflare(driver: uc.Chrome, timeout: int = CLOUDFLARE_TIMEOUT) -> bool:
    """Block until the Cloudflare challenge page disappears.

    Returns True when the real page has loaded, False on timeout.
    """
    try:
        WebDriverWait(driver, timeout).until_not(
            EC.title_contains("Just a moment...")
        )
        time.sleep(2)  # Allow the page scripts to finish initialising
        return True
    except Exception:
        print(f"❌ Cloudflare challenge not resolved within {timeout}s")
        return False


def collect_listing_urls(driver: uc.Chrome, url: str) -> list[str]:
    """Return all property listing hrefs found on *url*.

    Returns an empty list when the page fails to load or the expected
    elements are absent.
    """
    driver.get(url)
    if not wait_for_cloudflare(driver):
        return []

    try:
        WebDriverWait(driver, ELEMENT_TIMEOUT).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "js__product-link-for-product-id")
            )
        )
    except Exception:
        print(f"  ⚠️ Listing elements not found — page title: {driver.title!r}")
        return []

    elements = driver.find_elements(
        By.CLASS_NAME, "js__product-link-for-product-id"
    )
    return [el.get_attribute("href") for el in elements if el.get_attribute("href")]

## 7 · Phase 1 — Collect listing URLs

Crawls paginated listing pages until `TARGET_SAMPLES` URLs are gathered
or the last page is reached.  Results are written to `HREFS_FILE`.

In [14]:
driver = create_driver()
collected_urls: list[str] = []
page = 1

try:
    while len(collected_urls) < TARGET_SAMPLES:
        page_suffix = f"/p{page}" if page > 1 else ""
        url = f"{BASE_URL}{page_suffix}{QUERY_STRING}"
        print(f"📄 Page {page:>4} | {len(collected_urls):>5}/{TARGET_SAMPLES} collected — {url}")

        page_urls = collect_listing_urls(driver, url)
        remaining = TARGET_SAMPLES - len(collected_urls)
        collected_urls.extend(page_urls[:remaining])

        # Check whether a "next page" control exists
        next_buttons = driver.find_elements(
            By.CSS_SELECTOR,
            "a.re__pagination-icon:not(.re__pagination-icon--no-effect)",
        )
        if not next_buttons:
            print("🏁 Reached the last page.")
            break

        page += 1
        time.sleep(PAGE_DELAY)

finally:
    driver.quit()

# ── Sanity checks ────────────────────────────────────────────────────────────
assert len(collected_urls) > 0, "No URLs collected — check BASE_URL and network connectivity"
assert all(url.startswith("http") for url in collected_urls), "Some collected URLs look malformed"

print(f"\n✅ Collected {len(collected_urls)} URLs across {page} page(s)")

# ── Persist to disk ──────────────────────────────────────────────────────────
HREFS_FILE.write_text("\n".join(collected_urls), encoding="utf-8")
print(f"💾 Saved to {HREFS_FILE}")

📄 Page    1 |     0/1 collected — https://batdongsan.com.vn/ban-nha-dat-tp-hcm?vrs=1

✅ Collected 1 URLs across 2 page(s)
💾 Saved to hrefs.txt


## 8 · Phase 2 — Scrape property details

For each URL in `HREFS_FILE`, fetches and parses structured fields
(price, area, address, coordinates, legal status …) then appends a
row to `CSV_FILE` immediately so progress is never lost on interruption.

In [15]:
# ── Vietnamese field names as they appear in the target HTML ──────────────────
_FIELD_NAMES = (
    "Khoảng giá",
    "Diện tích",
    "Số phòng ngủ",
    "Số phòng tắm, vệ sinh",
    "Số tầng",
    "Hướng nhà",
    "Hướng ban công",
    "Đường vào",
    "Pháp lý",
    "Nội thất",
)


def scrape_property_detail(driver: uc.Chrome, url: str) -> dict:
    """Scrape structured data from a single property detail page.

    Returns a dict with fixed keys; missing values are ``numpy.nan``.
    """
    record: dict = {
        "link":         url,
        "address":      np.nan,
        "description":  np.nan,
        "latitude":     np.nan,
        "longitude":    np.nan,
        **{field: np.nan for field in _FIELD_NAMES},
    }

    try:
        driver.get(url)
        WebDriverWait(driver, ELEMENT_TIMEOUT).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "re__pr-other-info-display")
            )
        )
    except Exception:
        return record  # Page failed to load — return the empty record

    # Address
    try:
        record["address"] = driver.find_element(
            By.CLASS_NAME, "re__address-line-1"
        ).text.strip()
    except Exception:
        pass

    # GPS coordinates from the lazy-loaded map iframe
    try:
        data_src = driver.find_element(
            By.CSS_SELECTOR, "iframe.lazyload"
        ).get_attribute("data-src")
        lat, lon = data_src.split("q=")[1].split("&")[0].split(",")
        record["latitude"]  = lat.strip()
        record["longitude"] = lon.strip()
    except Exception:
        pass

    # Structured spec items (price, area, bedrooms, …)
    try:
        spec_items = driver.find_elements(
            By.CSS_SELECTOR,
            ".re__pr-other-info-display .re__pr-specs-content-item",
        )
        for item in spec_items:
            try:
                title = item.find_element(
                    By.CLASS_NAME, "re__pr-specs-content-item-title"
                ).text.strip()
                value = item.find_element(
                    By.CLASS_NAME, "re__pr-specs-content-item-value"
                ).text.strip()
                if title in record:
                    record[title] = value
            except Exception:
                continue
    except Exception:
        pass

    # Free-text description
    try:
        record["description"] = driver.find_element(
            By.CLASS_NAME, "re__section-body.re__detail-content"
        ).text.strip()
    except Exception:
        pass

    return record

In [16]:
# ── Load URL list ─────────────────────────────────────────────────────────────
assert HREFS_FILE.exists(), f"URL file not found: {HREFS_FILE} — run Phase 1 first"
urls = [u for u in HREFS_FILE.read_text(encoding="utf-8").splitlines() if u.strip()]
assert len(urls) > 0, f"{HREFS_FILE} is empty — re-run Phase 1"

total = len(urls)
print(f"📋 {total} URLs to process → {CSV_FILE}")

# ── Scrape loop ───────────────────────────────────────────────────────────────
driver = create_driver()

try:
    for idx, url in enumerate(urls, start=1):
        try:
            print(f"[{idx:>5}/{total}] {url}")
            record = scrape_property_detail(driver, url)
            row_df = pd.DataFrame([record])
            row_df.to_csv(
                CSV_FILE,
                mode="w" if idx == 1 else "a",
                header=(idx == 1),
                index=False,
                encoding="utf-8-sig",
            )
            time.sleep(DETAIL_DELAY)

        except Exception as exc:
            print(f"  ⚠️ Error on record {idx}: {exc} — restarting driver")
            driver.quit()
            driver = create_driver()

finally:
    driver.quit()

print(f"\n✅ Done — results saved to {CSV_FILE}")

📋 1 URLs to process → data_bds.csv
[    1/1] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-duong-nguyen-xien-phuong-long-binh-3-the-manhattan-glory-vinhomes-grand-park/chinh-chu-gui-ban-lo-pho-truc-sinh-23-9-ty-144-m2-so-hong-pr45709145


KeyboardInterrupt: 

## 9 · Phase 3 — Extract features from description text *(optional)*

Parses free-text listing descriptions to recover attributes not exposed
in structured form: dimensions, amenity flags, urgency signals, etc.

In [ ]:
_DIRECTION_PATTERN = (
    r"(đông[\s-]?nam|đông[\s-]?bắc|tây[\s-]?nam|tây[\s-]?bắc"
    r"|đông|tây|nam|bắc)"
)

_LEGAL_STATUS_MAP: dict[str, str] = {
    "sổ hồng":           "Sổ hồng",
    "sổ đỏ":             "Sổ đỏ",
    "shr":               "Sổ hồng",
    "so hong":           "Sổ hồng",
    "hợp đồng mua bán":  "HĐMB",
    "hdmb":              "HĐMB",
    "giấy tờ tay":       "Giấy tay",
    "giấy tay":          "Giấy tay",
    "chờ sổ":            "Chờ sổ",
    "đang làm sổ":       "Chờ sổ",
}

_AMENITY_PATTERNS: dict[str, str] = {
    "near_market":       r"(gần chợ|cạnh chợ|sát chợ|chợ \w{1,20}\s*(\d+m|\d+ m)?)",
    "near_school":       r"(gần trường|cạnh trường|trường học|trường tiểu học|thpt|thcs|mẫu giáo)",
    "near_hospital":     r"(bệnh viện|bv\b|phòng khám)",
    "near_supermarket":  r"(siêu thị|vinmart|coopmart|lotte|bigc|aeon|go!|tops market)",
    "near_mall":         r"(trung tâm thương mại|tttm|vincom|nowzone|gigamall)",
    "near_park":         r"(công viên|cv\b|cây xanh|hồ bơi công cộng)",
    "near_river":        r"(sông|kênh|rạch|bờ sông|view sông|ven sông)",
    "near_airport":      r"(sân bay|tsn|tân sơn nhất)",
    "near_station":      r"(bến xe|nhà ga|metro|tàu điện|ga tàu)",
    "near_industrial":   r"(khu công nghiệp|kcn\b|khu chế xuất|kcx\b)",
    "near_university":   r"(đại học|cao đẳng|trường đh|trường cđ)",
    "near_bank":         r"(ngân hàng|bank|atm)",
    "near_pool":         r"(hồ bơi|bể bơi|swimming pool)",
    "near_gym":          r"(gym|phòng gym|thể dục|thể thao|fitness)",
    "near_temple":       r"(nhà thờ|chùa|đình|miếu)",
    "near_bridge":       r"(cầu \w{1,20}|gần cầu)",
    "private_pool":      r"(hồ bơi riêng|hồ bơi nội khu|hồ bơi gia đình)",
    "in_compound":       r"(khu dân cư|kdc|vinhomes|gamuda|nam long|phú mỹ hưng|the manor|sala|eco)",
    "security_24h":      r"(an ninh|bảo vệ 24|camera an ninh|security|bảo vệ)",
    "near_beach":        r"(biển|bãi biển|bờ biển|view biển)",
}


def parse_description(description: str) -> dict:
    """Extract structured features from a free-text property description.

    Parameters
    ----------
    description:
        Raw listing description text (Vietnamese).

    Returns
    -------
    dict
        Extracted features.  Absent values are ``numpy.nan``.
    """
    if not isinstance(description, str) or not description.strip():
        return {}

    text = description.lower()
    result: dict = {}

    # ── Area & dimensions ────────────────────────────────────────────────────
    m = re.search(
        r"di[eệ]n t[ií]ch(?:\s+(?:đ[aấ]t|khu[ôô]n đ[aấ]t|thực tế))?[:\s]*(\d+[,.]?\d*)\s*m",
        text,
    )
    result["land_area_desc"] = m.group(1).replace(",", ".") if m else np.nan

    m = re.search(
        r"di[eệ]n t[ií]ch\s*(?:xây dựng|sàn|sd|sử dụng)[:\s]*(\d+[,.]?\d*)\s*m",
        text,
    )
    result["floor_area_desc"] = m.group(1).replace(",", ".") if m else np.nan

    m = re.search(
        r"(?:ngang[:\s]*)?\(?(\d+[,.]?\d*)\s*[x×]\s*(\d+[,.]?\d*)\s*m?\)?",
        text,
    )
    if m:
        result["width_desc"]  = m.group(1).replace(",", ".")
        result["length_desc"] = m.group(2).replace(",", ".")
    else:
        result["width_desc"]  = np.nan
        result["length_desc"] = np.nan

    m = re.search(
        r"(?:m[aặ]t ti[eề]n|ngang)[:\s]*(\d+[,.]?\d*)\s*m",
        text,
    )
    result["frontage_desc"] = m.group(1).replace(",", ".") if m else np.nan

    # ── Floors & rooms ───────────────────────────────────────────────────────
    m = re.search(r"(\d+[,.]?\d*)\s*t[aầ]ng", text)
    result["floors_desc"] = m.group(1) if m else np.nan

    m = re.search(r"(\d+)\s*(?:phòng ngủ|pn|p\.ngủ|p ngủ|bedroom)", text)
    result["bedrooms_desc"] = m.group(1) if m else np.nan

    m = re.search(r"(\d+)\s*(?:toilet|wc|nhà vệ sinh|phòng tắm|pt)", text)
    result["bathrooms_desc"] = m.group(1) if m else np.nan

    result["has_basement"]  = "yes" if re.search(r"(t[aầ]ng h[aầ]m|h[aầ]m xe|basement)", text) else "no"
    result["has_elevator"]  = "yes" if re.search(r"thang m[aá]y|elevator|lift", text) else "no"
    result["has_rooftop"]   = "yes" if re.search(r"s[aâ]n th[ưượ]ng|rooftop|sân thượng", text) else "no"

    # ── Price ────────────────────────────────────────────────────────────────
    m = re.search(
        r"gi[aá](?:\s+b[aá]n)?[:\s]*(\d+[,.]?\d*)\s*(t[yỷ]|tri[eệ]u|tr\.?|tỉ)",
        text,
    )
    if m:
        unit = "tỷ" if m.group(2)[0] == "t" else "triệu"
        result["price_desc"] = f"{m.group(1).replace(',', '.')} {unit}"
    else:
        result["price_desc"] = np.nan

    m = re.search(r"(\d+[,.]?\d*)\s*(?:tri[eệ]u|tr)\.?\s*/\s*m", text)
    result["price_per_sqm_desc"] = (
        m.group(1).replace(",", ".") + " tr/m²" if m else np.nan
    )

    # ── Orientation ──────────────────────────────────────────────────────────
    m = re.search(
        r"hướng(?:\s+(?:cửa chính|nhà|chính))?[:\s]*" + _DIRECTION_PATTERN,
        text,
    )
    result["house_facing_desc"] = m.group(1).title() if m else np.nan

    m = re.search(r"hướng ban công[:\s]*" + _DIRECTION_PATTERN, text)
    result["balcony_facing_desc"] = m.group(1).title() if m else np.nan

    # ── Legal status ─────────────────────────────────────────────────────────
    result["legal_status_desc"] = np.nan
    for keyword, label in _LEGAL_STATUS_MAP.items():
        if keyword in text:
            result["legal_status_desc"] = label
            break

    # ── Furnishing ───────────────────────────────────────────────────────────
    if re.search(r"full n[oộ]i th[aấ]t|đầy đủ n[oộ]i th[aấ]t|n[oộ]i th[aấ]t cao c[aấ]p", text):
        result["furnishing_desc"] = "full"
    elif re.search(r"n[oộ]i th[aấ]t c[oơ] b[aả]n|c[oơ] b[aả]n", text):
        result["furnishing_desc"] = "basic"
    elif re.search(r"không n[oộ]i th[aấ]t|bàn giao thô|nhà thô", text):
        result["furnishing_desc"] = "none"
    else:
        result["furnishing_desc"] = np.nan

    # ── Road / alley type ────────────────────────────────────────────────────
    result["car_accessible_desc"] = "yes" if re.search(
        r"(hẻm xe (hơi|tải|ô tô)|ô tô ngủ trong|xe tải|xe hơi vào|xe 7 chỗ|oto)",
        text,
    ) else "no"

    if re.search(r"m[aặ]t ti[eề]n|mt\b|mặt phố", text):
        result["road_type_desc"] = "main_street"
    elif re.search(r"hxh|hẻm xe hơi", text):
        result["road_type_desc"] = "car_alley"
    elif re.search(r"hẻm", text):
        result["road_type_desc"] = "alley"
    else:
        result["road_type_desc"] = np.nan

    m = re.search(r"(?:đường|hẻm|lộ giới|lg)[:\s]*(\d+[,.]?\d*)\s*m", text)
    result["road_width_desc"] = m.group(1).replace(",", ".") if m else np.nan

    # ── Amenity flags ────────────────────────────────────────────────────────
    for flag, pattern in _AMENITY_PATTERNS.items():
        result[flag] = "yes" if re.search(pattern, text) else "no"

    # ── Seller intent signals ─────────────────────────────────────────────────
    result["urgent_sale"]   = "yes" if re.search(
        r"(b[aá]n g[aấ]p|kẹt ti[eề]n|cần ti[eề]n|bán lỗ|bán nhanh|ưu ti[eê]n)",
        text,
    ) else "no"
    result["negotiable"]    = "yes" if re.search(
        r"(thương lượng|tl\b|có thể tl|xem xét)", text
    ) else "no"
    result["owner_selling"] = "yes" if re.search(
        r"(chính chủ|chủ nhà|chủ bán)", text
    ) else "no"

    # ── Suitability flags ────────────────────────────────────────────────────
    result["suitable_for_business"]  = "yes" if re.search(
        r"(kinh doanh|buôn bán|cho thuê|mặt bằng kd)", text
    ) else "no"
    result["suitable_for_living"]    = "yes" if re.search(
        r"(để ở|an cư|vào ở ngay|ở ngay|phù hợp ở)", text
    ) else "no"
    result["suitable_for_investing"] = "yes" if re.search(
        r"(đầu tư|sinh lời|dòng ti[eề]n|cho thuê|roi\b)", text
    ) else "no"

    return result

In [ ]:
# ── Apply to full dataset ─────────────────────────────────────────────────────
assert CSV_FILE.exists(), f"CSV file not found: {CSV_FILE} — run Phase 2 first"

df = pd.read_csv(CSV_FILE, encoding="utf-8-sig")
assert "description" in df.columns, "Expected a 'description' column in the CSV"

desc_features = pd.DataFrame(
    df["description"].apply(parse_description).tolist()
)
df_expanded = pd.concat([df, desc_features], axis=1)

output_path = CSV_FILE.with_name(CSV_FILE.stem + "_expanded.csv")
df_expanded.to_csv(output_path, index=False, encoding="utf-8-sig")

# ── Output sanity checks ──────────────────────────────────────────────────────
assert len(df_expanded) == len(df), "Row count changed after description parsing — data integrity issue"
assert "land_area_desc" in df_expanded.columns, "Description parsing produced no new columns"

print(f"✅ Expanded dataset saved to {output_path}")
print(f"   Rows    : {len(df_expanded)}")
print(f"   Columns : {len(df_expanded.columns)} ({len(df.columns)} original + {len(desc_features.columns)} new)")
print(df_expanded.head(3).to_string())